# State Space Model (SSM) 공부 노트
> Mamba 이해를 위한 기초 정리

## 1. SSM이란?

> **상태방정식 + 출력방정식** 두 개를 합친 시스템

- 상태방정식만으로는 $x(t)$가 어떻게 변하는지만 알 수 있음
- 출력방정식이 있어야 $x(t)$를 $y(t)$로 꺼낼 수 있음
- 두 방정식이 하나의 시스템으로 묶인 것이 SSM

---

## 2. SSM 수식 표현

### 연속 시간

$$\dot{x}(t) = Ax(t) + Bu(t)$$
$$y(t) = Cx(t) + Du(t)$$

### 이산 시간 (딥러닝에서 실제 사용)

$$x_k = \bar{A}x_{k-1} + \bar{B}u_k$$
$$y_k = Cx_k$$

> 컴퓨터는 연속 시간을 다룰 수 없으므로, 시간 간격 $\Delta$로 이산화해서 사용한다.

```
입력 u(t) ──┬──[B]──→ ⊕ ──→ x(t) ──[C]──→ ⊕──→ 출력 y(t)
            │         ↑                     ↑
            │        [A]←──────────────────┘
            └──────────────[D]─────────────┘
```

## 내가 직접 정리한 한 문장 정의

**SSM:**
> "과거의 입력들을 압축해서 현재를 판단하는 기계다. 표현은 상태방정식과 출력방정식으로 표현된다."

**HiPPO:**
> "A와 x(t) 값이 있는데 x(t)의 과거 정보들을 압축한 문맥정보를 A에 의해서 길거나 순차적인 정보 중 훨씬 과거의 기록들의 정보가 한번에 사라지지 않지만 옅은 값으로 표현되어 있어 장기적인 정보로 일부는 살아남는다."

**SSM vs HiPPO 핵심 차이:**
> "SSM은 A 행렬을 랜덤으로 값을 정하다 보니 전체 흐름(방향)이 이리저리 뭉개질 수 있다. 대신 HiPPO는 A 행렬을 수학적으로 틀을 만들어두고 기본 방향성을 평균적인 흐름으로 세부적인 조정을 하고 세부적인 내용이 근본이 되는 $a_0$을 흔들지 않는다. 이것이 하삼각 행렬 (Lower Triangular Matrix)로 보장된다."

## 2. 각 요소의 정체

| 기호 | 정체 | 시간에 따라 변하나? |
|---|---|---|
| $u(t)$ | 내가 넣는 입력 데이터 | ✅ 매 스텝 다름 |
| $y(t)$ | 모델이 뱉는 출력 데이터 | ✅ 매 스텝 다름 |
| $A, B, C$ | 모델의 가중치 | ❌ 학습 후 고정 |
| $x(t)$ | **모델이 내부적으로 유지하는 압축된 맥락 벡터** | ✅ 매 스텝 업데이트 |

### x(t)를 헷갈리지 않으려면
- 데이터($u, y$)도 아님
- 모델 가중치($A,B,C$)도 아님
- 가중치가 입력을 처리하면서 **만들어내는 결과물** — 과거 입력들의 압축된 기억
- RNN의 $h_t$와 동일한 역할

## 3. 이산화 (컴퓨터에서 사용하려면)

연속 시간 → 시간 간격 $\Delta$로 이산화

$$x_k = \bar{A}x_{k-1} + \bar{B}u_k$$
$$y_k = Cx_k$$

| 연속 | 이산 |
|---|---|
| $A$ | $\bar{A} = e^{\Delta A}$ |
| $B$ | $\bar{B} = (\Delta A)^{-1}(e^{\Delta A}-I)\Delta B$ |
| (없음) | $\Delta$ ← Mamba의 핵심 파라미터 |

## 4. 전통 SSM의 한계 → Mamba로 이어지는 이유

**$A$ 행렬이 고정** = 입력 내용에 상관없이 항상 같은 비율로 기억하고 잊음

현실에서는 중요한 내용은 강하게 기억하고, 덜 중요한 건 흘려보내야 함 → **선택적 집중 불가**

**Mamba의 해결:**
$$\Delta, B, C = f(u_t) \quad \leftarrow \text{입력에 따라 동적으로 변화 = Selective SSM}$$

## 5. 코드로 확인하기

개념이 흔들릴 때는 코드로 직접 확인한다.

In [2]:
# SSM 이산화 기본 동작
# x(t) = A·x(t-1) + B·u(t)
# y(t) = C·x(t)

A = 0.99   # 과거 맥락 유지 비율 (1에 가까울수록 오래 기억)
B = 0.1   # 새 입력 흡수 비율
C = 1.0   # 출력 변환

x = 0.0   # 은닉 상태 초기값

print(f"{'입력 u':>8} {'은닉 x':>10} {'출력 y':>10}")
print("-" * 32)

for u in [1, 2, 3, 4, 5]:
    x = A * x + B * u    # 상태 방정식 (이산)
    y = C * x            # 출력 방정식
    print(f"{u:>8}   {x:>10.4f}   {y:>10.4f}")

    입력 u       은닉 x       출력 y
--------------------------------
       1       0.1000       0.1000
       2       0.2990       0.2990
       3       0.5960       0.5960
       4       0.9900       0.9900
       5       1.4801       1.4801


## 6. 학습 로드맵

```
상태방정식 + 출력방정식  ← 현재
        ↓
    SSM (두 방정식의 통합)  ← 현재
        ↓
   HiPPO (A 행렬을 수학적으로 설계)
        ↓
   S4 (SSM을 병렬 연산 가능하게 변환)
        ↓
   Mamba (Selective SSM — B, C, Δ를 입력에 따라 동적 변화)
```

## HiPPO

### 풀네임
**Hi**gh-order **P**olynomial **P**rojection **O**perator (고차 다항식 투영 연산자)

### 한 문장 정의
> 지금까지 들어온 입력 시퀀스 전체를, 고차 다항식으로 근사해서 압축하는 연산자

---

### SSM vs HiPPO

| | 일반 SSM | HiPPO |
|---|---|---|
| $A$ 초기값 | 랜덤 | 수학적으로 설계된 값 |
| $A$ 학습 | 역전파로 업데이트 | 고정 또는 구조 유지 |
| 장기 기억 | 보장 없음 | 수학적으로 보장 |
| 구조 | 아무 형태 | 하삼각 행렬 |

> 핵심: SSM은 단순히 랜덤값이 들어있고 역전파로 업데이트되지만, HiPPO는 무엇을 기억하고 버릴지 수학적으로 설계되어 장기 기억을 보장한다.

---

### 다항식 투영이 왜 기억에 좋은가

복잡한 곡선 전체를 계수 N개로 표현할 수 있다.

| | 단순 저장 | 다항식 투영 |
|---|---|---|
| 저장 공간 | 입력 개수만큼 | 계수 N개 고정 |
| 오래된 기억 | 덮어쓰여 사라짐 | 곡선에 반영됨 |
| 전체 흐름 | 파악 불가 | 계수로 요약됨 |

- **다항식 계수 = 은닉 상태 $x$**
- $x$가 업데이트되는 것 = 다항식 계수가 갱신되는 것
- N은 표현력과 계산 비용의 트레이드오프로 실험적으로 결정 (보통 64, 128, 256)

---

### HiPPO $A$ 행렬 (HiPPO-LegS)

**조건 1. 대각선 아래 ($n > k$)**

$$A_{nk} = -(2n+1)^{1/2}(2k+1)^{1/2}$$

**조건 2. 대각선 ($n = k$)**

$$A_{nk} = -(n+1)$$

**조건 3. 대각선 위 ($n < k$)**

$$A_{nk} = 0$$

**실제 모양 (N=4)**

$$A = -\begin{bmatrix} 1 & 0 & 0 & 0 \\ \sqrt{3} & 2 & 0 & 0 \\ \sqrt{5} & \sqrt{15} & 3 & 0 \\ \sqrt{7} & \sqrt{21} & \sqrt{35} & 4 \end{bmatrix}$$

---

### 각 계수의 역할

| 행 ($n$) | 계수 | 의미 | 대각선 (감쇠 속도) |
|---|---|---|---|
| $n=0$ | $a_0$ | 전체 평균, 큰 흐름 | $-1$ (천천히 감쇠) |
| $n=1$ | $a_1$ | 선형 추세 | $-2$ |
| $n=2$ | $a_2$ | 곡률 | $-3$ |
| $n=3$ | $a_3$ | 세밀한 패턴 | $-4$ (빠르게 감쇠) |

---

### $n=3$ 행 업데이트 식

$$a_3^{new} = -\sqrt{7}\cdot a_0 - \sqrt{21}\cdot a_1 - \sqrt{35}\cdot a_2 - 4\cdot a_3 + \sqrt{7}\cdot u$$

- 가까운 차수일수록 영향이 강함 ($\sqrt{7} < \sqrt{21} < \sqrt{35}$)
- 대각선 값이 클수록 빠르게 감쇠

---

### 왜 장기 기억인가

> $x(t)$는 과거 입력들의 압축된 문맥이고, $A$의 대각선 설계 덕분에 오래된 정보는 완전히 사라지지 않고 **점점 옅어지면서 낮은 차수 계수($a_0$)에 흔적으로 남는다.**

| | 오래된 정보 처리 |
|---|---|
| 일반 RNN | 기울기 소실 → 완전히 사라짐 |
| HiPPO | 옅어지지만 $a_0$에 흔적으로 살아남음 |

## HiPPO 심화 — 헷갈렸지만 이해한 것들

### 왜 $a_0$가 평균인가

4개의 점 $[1, 3, 2, 4]$를 **수평선 하나**로 표현할 때, 모든 점과의 오차가 가장 작은 수평선은 어디인가?

| 수평선 위치 | 오차 합 |
|---|---|
| 4 (최댓값) | 3+1+2+0 = 6 |
| 2.5 (평균) | 1.5+0.5+0.5+1.5 = 4 ← 최소 |

**오차를 최소화하는 수평선 = 평균 = $a_0$**

0차 다항식은 수평선이고, 수평선이 데이터를 가장 잘 표현하는 위치는 수학적으로 평균이다.

---

### 다항식 근사의 순서

$a_0$ 하나만으로 설명 못 하는 나머지 오차를 $a_1, a_2, a_3$가 순서대로 줄여나간다.

```
1단계: a₀ (수평선)  → 전체 평균, 가장 큰 오차 감소
2단계: a₁ 추가     → 추세 반영, 오차 더 감소
3단계: a₂ 추가     → 곡률 반영, 오차 더 감소
4단계: a₃ 추가     → 세부패턴 반영, 오차 더 감소
```

> $a_0$는 "일단 평균으로 설명하고", 나머지 계수들이 "평균으로 설명 못 한 부분을 순서대로 채워나간다."

---

### 다항식 계수 N vs 채널 수

| | 다항식 계수 | 채널 수 |
|---|---|---|
| 개수 | N개 고정 (하이퍼파라미터) | N개 고정 |
| 역할 결정 | 수학이 미리 정함 ($a_0$=평균 등) | 학습이 알아서 결정 |
| 장기 기억 | 수학적으로 보장 | 보장 없음 |

> 채널은 학습이 역할을 만들고, 다항식 계수는 수학이 역할을 미리 정한다.

---

### 다항식 계수 N vs 문장 길이

| | 의미 | 누가 정하나 |
|---|---|---|
| 문장 길이 | 단어 몇 개짜리 문장 | 데이터에 따라 다름 |
| 계수 N | 은닉 상태 $x$의 차원 | 우리가 정하는 하이퍼파라미터 |

문장이 100단어든 10단어든, $x$는 항상 N차원 벡터로 고정.

> 페이지가 몇 장이든 요약본 항목 수는 우리가 미리 정한 대로 고정.

## SSM 계산 흐름 — $\dot{x}(t)$에서 $x(t)$를 어떻게 구하나

### 핵심 질문

상태방정식 $\dot{x}(t) = Ax(t) + Bu(t)$ 는 $x$의 **변화율**을 주지, $x$ 자체를 주지 않는다.

### 연속 시간 — 적분으로 구함

$$x(t) = x(0) + \int_0^t (Ax(\tau) + Bu(\tau))d\tau$$

컴퓨터는 적분을 직접 계산할 수 없으므로 이산화가 필요하다.

### 이산 시간 — 이전 x로 다음 x를 계산

$$\dot{x} \approx \frac{x_k - x_{k-1}}{\Delta} \quad \Rightarrow \quad x_k = \bar{A}x_{k-1} + \bar{B}u_k$$

```
초기값: x₀ = 0

스텝 1: u₁ 입력 → x₁ = Ā·x₀ + B̄·u₁ → y₁ = C·x₁
스텝 2: u₂ 입력 → x₂ = Ā·x₁ + B̄·u₂ → y₂ = C·x₂
스텝 3: u₃ 입력 → x₃ = Ā·x₂ + B̄·u₃ → y₃ = C·x₃
```

매 스텝마다 직전 $x$가 있으니까 다음 $x$를 구할 수 있고, $y$도 바로 나온다. 이게 RNN과 동일한 구조인 이유다.

---

## 행렬 차원 (N=4일 때)

| | 차원 |
|---|---|
| $u_k$ | 1×1 (입력 하나) |
| $x_k$ | **N×1** (계수 N개 벡터) |
| $y_k$ | 1×1 (출력 하나) |
| $A$ | N×N |
| $B$ | N×1 |
| $C$ | 1×N |

$x$가 하나의 값이 아니라 **N개의 계수를 동시에 담은 벡터**인 것이 정상이다.

$$\underbrace{(N\times1)}_{x_k} = \underbrace{(N\times N)}_{\bar{A}}\underbrace{(N\times1)}_{x_{k-1}} + \underbrace{(N\times1)}_{\bar{B}}\underbrace{(1\times1)}_{u_k}$$

---

## 연구와의 연결 (ET-NAGraphSAGE)

```
u(t) = T프레임의 노드/엣지 피처 (속도, 가속도, 상대거리 등)
x(t) = 과거 T프레임이 압축된 차량 맥락 벡터
y(t) = 현재 차량 상태 (Stop / Lane Change / Normal Driving)
A, B = Mamba/GRU/LSTM의 가중치
```

| SSM | 연구 |
|---|---|
| $u(t)$ | T프레임 노드/엣지 피처 |
| $x(t)$ | 압축된 차량 맥락 벡터 |
| $y(t)$ | Stop / Lane Change / Normal |
| $A, B$ | Mamba/GRU/LSTM 가중치 |

NAGraphSAGE는 현재 프레임 1개만 보지만, ET-NAGraphSAGE는 T프레임 맥락이 $x(T)$에 압축되어 있어 판단이 더 정확하다.

## S4 (Structured State Space Sequence Model)

### 한 문장 정의
> HiPPO로 설계한 A 행렬을 빠르게 연산할 수 있도록 구조화한 모델

### 왜 S4가 필요했나

순환 방식은 순서대로 하나씩 계산해야 해서 GPU 병렬 연산을 못 씀.

```
순환:  x₁→x₂→x₃ ... (기다려야 함)
S4:   K를 미리 계산 → 입력 전체와 한 번에 합성곱 (병렬)
```

---

### 커널이 어디서 오는가

이산 SSM 반복식을 풀어쓰면:

$$x_1 = \bar{B}u_1$$
$$x_2 = \bar{A}\bar{B}u_1 + \bar{B}u_2$$
$$x_3 = \bar{A}^2\bar{B}u_1 + \bar{A}\bar{B}u_2 + \bar{B}u_3$$

> **Q: $x_1 = \bar{B}u_1$ 에서 A 행렬은 왜 없나?**
>
> $k=1$ 대입: $x_1 = \bar{A}x_0 + \bar{B}u_1$
> 초기값 $x_0 = 0$ 이므로: $x_1 = \bar{A}\cdot 0 + \bar{B}u_1 = \bar{B}u_1$
>
> $\bar{A}$가 없는 게 아니라, **$x_0 = 0$ 이라 $\bar{A}$ 항이 사라진 것.**
> $k=2$부터는 $x_1$이 존재하니 $\bar{A}$가 등장한다.
> $$x_2 = \bar{A}x_1 + \bar{B}u_2 = \bar{A}\bar{B}u_1 + \bar{B}u_2$$

출력 $y_k = Cx_k$ 를 대입하면:

$$y_1 = C\bar{B} \cdot u_1$$
$$y_2 = C\bar{A}\bar{B} \cdot u_1 + C\bar{B} \cdot u_2$$
$$y_3 = C\bar{A}^2\bar{B} \cdot u_1 + C\bar{A}\bar{B} \cdot u_2 + C\bar{B} \cdot u_3$$

패턴으로 정리하면:

$$K = (C\bar{B},\ C\bar{A}\bar{B},\ C\bar{A}^2\bar{B},\ \ldots,\ C\bar{A}^{T-1}\bar{B})$$

| 커널 값 | 의미 |
|---|---|
| $C\bar{B}$ | 바로 직전 입력의 영향력 |
| $C\bar{A}\bar{B}$ | 2스텝 전 입력의 영향력 |
| $C\bar{A}^{t-1}\bar{B}$ | t스텝 전 입력의 영향력 |

---

### 숫자 예시 (Ā=0.9, B̄=0.1, C=1.0, T=4)

$$K = (0.1,\ 0.09,\ 0.081,\ 0.073)$$

$$y_4 = 0.073\cdot u_1 + 0.081\cdot u_2 + 0.09\cdot u_3 + 0.1\cdot u_4$$

과거로 갈수록 커널 값이 작아짐 → 오래된 입력의 영향이 줄어듦

---

### K를 미리 계산할 수 있는 이유

| 행렬 | 결정 시점 | 이유 |
|---|---|---|
| $A$ | N 정하면 즉시 | HiPPO 공식으로 계산 |
| $B$ | N 정하면 즉시 | HiPPO 공식으로 계산 |
| $C$ | 학습으로 결정 | 태스크에 따라 다름 |
| $\Delta$ | 학습으로 결정 | 이산화 간격 |

> HiPPO 덕분에 A, B가 고정 → K를 미리 계산 가능 → 입력과 합성곱으로 한 번에 처리
> 순환: T번 순차 계산 / S4: FFT로 O(T log T) 병렬 계산

---

## RNN vs HiPPO 장기 기억 비교 ⭐️ 중요

### RNN의 장기 기억 문제 — 기울기 소실

$$h_t = \tanh(W_h \cdot h_{t-1} + W_x \cdot u_t)$$

역전파 시 오래된 시점까지 기울기 전달:

$$\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial h_t} \cdot \prod_{k=2}^{t} \frac{\partial h_k}{\partial h_{k-1}}$$

```
T=10이면: W_h를 10번 곱함
  W_h < 1 → 0에 수렴 → 기울기 소실
  W_h > 1 → ∞에 발산 → 기울기 폭발
```

결과: 10프레임 전 정보는 학습이 안 됨

### HiPPO는 왜 다른가

```
RNN:   A를 역전파로 학습 → 기울기 소실로 장기 기억 실패
HiPPO: A를 수학으로 설계 → 역전파 없이도 장기 기억 보장
```

### 핵심 비교표

| | RNN | HiPPO |
|---|---|---|
| $A$ 결정 방식 | 역전파로 학습 | 수학적으로 고정 설계 |
| 장기 기억 | 기울기 소실로 실패 | $a_0$ 감쇠 -1로 보장 |
| 오래된 정보 | 완전히 사라짐 | 옅어지지만 $a_0$에 남음 |
| 학습 안정성 | 불안정 | 안정 |

## RNN의 tanh와 기울기 소실

### tanh를 도입한 이유

tanh 없이 순수 선형이면:
$$h_t = W_h \cdot h_{t-1} + W_x \cdot u_t$$

```
W_h = 2 이면: 2 → 4 → 8 → 16 → ... 무한 발산
W_h = 0.5이면: 0.5 → 0.25 → ... → 0 소실
```

tanh는 출력을 **-1 ~ +1 사이로 강제 압축**해서 값 폭발을 막고 비선형성을 도입한다.

### tanh의 한계 — 기울기 소실 계산

$$\tanh'(x) = 1 - \tanh^2(x)$$

| 입력 x | tanh(x) | tanh'(x) |
|---|---|---|
| 0 | 0.0 | 1.0 (최대) |
| 1 | 0.76 | 0.42 |
| 2 | 0.96 | 0.07 |
| 3 | 0.995 | 0.01 (거의 0) |

### 역전파 기울기 계산 (T=4, $W_h=0.8$, 입력 포화 가정)

각 스텝: $\tanh(2.0) = 0.964$, $\tanh'(2.0) = 1 - 0.964^2 = 0.071$

$$\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial h_4} \cdot (0.071 \times 0.8)^3 = \frac{\partial L}{\partial h_4} \cdot 0.057^3 = \frac{\partial L}{\partial h_4} \cdot 0.000185$$

### T=10이면

$$\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial h_{10}} \cdot (0.057)^9 \approx \frac{\partial L}{\partial h_{10}} \cdot 0.000000005$$

```
h₁₀→h₉ →h₈ →h₇ →h₆ →h₅ →h₄ →h₃ →h₂ →h₁
   ×0.057×0.057×0.057×0.057×0.057×0.057×0.057×0.057×0.057

기울기: 1.0 → 0.057 → 0.003 → 0.0002 → ≈ 0
```

> tanh 자체가 문제가 아니라, **tanh'이 항상 1보다 작고** 이걸 T번 곱하면 지수적으로 0에 수렴하는 것이 문제.

### 정리

| | 역할 | 한계 |
|---|---|---|
| tanh | 값 폭발 방지, 비선형성 도입 | 기울기 소실은 해결 못함 |
| LSTM gate | 기울기 소실 일부 완화 | 완전 해결 아님 |
| HiPPO | A를 수학적 설계 | 역전파 없이 장기 기억 보장 |